# 04 · Time-resolved mixed-effects models and figures

For each measure, trial-level values are averaged in consecutive 20-ms bins (10–11 samples at 512 Hz)
and a linear mixed-effects model is fitted separately for each bin:

`measure ~ 1 + memorability`, with a random intercept and a random memorability slope per contact,

fitted with [GPBoost](https://github.com/fabsig/GPBoost). The memorability coefficient is tested with a
two-sided Wald z-test. Before fitting, values below the 1st or above the 99th percentile within each bin are
excluded. P-values are FDR-corrected (Benjamini–Hochberg) across the bins of each measure; corrected
p < 0.05 is considered significant.

* Theta measures: 50 bins, 0–1000 ms after image onset.
* ERP: 60 bins, −200–1000 ms, after subtracting each trial's mean over −200–0 ms.

Figures 2–5 of the paper.
Composite figures were assembled in Inkscape from the panels saved here.

Input: `derivatives/features/`. Outputs: `results/gpboost_results.csv`, `results/significant_windows.csv`, `figures/`.

To reproduce the results from the shared dataset, run `06_features_from_shared_data.ipynb` and then this notebook
with `USE_SHARED_DATA=1`. It then reads `derivatives/features_from_shared_data/` and writes to
`results/from_shared_data/` and `figures/from_shared_data/`; measures that are not in the shared data
(the ERP, if the broadband LFP was not exported) are skipped.

In [ ]:
import pickle

import gpboost as gpb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests

import config

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SFREQ = config.SFREQ
ONSET = int(round(-config.EPOCH_TMIN * SFREQ))     # 256: sample of image onset in the stored arrays
BASELINE_START = ONSET - 102                       # 154: ~-200 ms
SAMPLES_PER_BIN = 20 / (1000 / SFREQ)              # 10.24 samples per 20-ms bin
ALPHA = 0.05

## Load features

Contact-trials with missing data (all-zero LFP between −500 and 1000 ms, see notebook 02) are excluded.
Features computed from the shared dataset carry this information as a `missing` flag.

In [ ]:
SINGLE_BAND = ["lfp"]                           # (trials, 1, time)
THETA = ["power", "phaselock", "speed"]         # (trials, 2, time): 0 = 3.5 Hz -> *_low, 1 = 7 Hz -> *_high
features_dir = config.SHARED_FEATURES_DIR if config.USE_SHARED_DATA else config.FEATURES_DIR
print(f"Reading features from {features_dir}")

frames = []
n_total = 0
for path in sorted(features_dir.glob("Hippocampus_*.pkl")):
    _, subject, contact = path.stem.split("_")
    with open(path, "rb") as f:
        features = pickle.load(f)

    df = pd.DataFrame({"memorability": features["memorability"], "subjectID": subject, "electrode": contact})
    for key in SINGLE_BAND:
        if key in features:
            df[key] = list(features[key][:, 0, :])
    for key in THETA:
        df[f"{key}_low"] = list(features[key][:, 0, :])
        df[f"{key}_high"] = list(features[key][:, 1, :])

    n_total += len(df)
    if "missing" in features:  # shared dataset
        has_data = ~np.asarray(features["missing"], dtype=bool)
    else:
        has_data = np.array([not (np.isnan(x).any() or np.all(x == 0)) for x in df["lfp"]])
    frames.append(df[has_data])

full_data = pd.concat(frames, ignore_index=True)
full_data["subject_electrode"] = full_data["subjectID"] + ":" + full_data["electrode"]

print(f"{full_data['subjectID'].nunique()} participants, {full_data['subject_electrode'].nunique()} contacts")
print(f"Contact-trials: {n_total}, after excluding missing data: {len(full_data)} "
      f"({n_total - len(full_data)} excluded)")

## Model fitting

In [ ]:
def bin_feature(full_data, col, include_pre_stimulus=False):
    """Long-format table with the mean of `col` in consecutive 20-ms bins.

    include_pre_stimulus=False: 50 bins from image onset (0-1000 ms).
    include_pre_stimulus=True: 60 bins from -200 ms, after subtracting each trial's -200-0 ms mean.
    """
    n_bins, start = (60, BASELINE_START) if include_pre_stimulus else (50, ONSET)
    edges = np.round(np.arange(n_bins + 1) * SAMPLES_PER_BIN).astype(int)
    end = start + int(np.round(n_bins * SAMPLES_PER_BIN))

    mat = np.vstack(full_data[col].values)
    window = mat[:, start:end]
    if include_pre_stimulus:
        window = window - mat[:, BASELINE_START:ONSET].mean(axis=1, keepdims=True)
    binned = np.add.reduceat(window, edges[:-1], axis=1) / np.diff(edges)

    n_rows = len(full_data)
    return pd.DataFrame({
        "memorability": np.repeat(full_data["memorability"].to_numpy(), n_bins),
        "subjectID": np.repeat(full_data["subjectID"].to_numpy(), n_bins),
        "subject_electrode": np.repeat(full_data["subject_electrode"].to_numpy(), n_bins),
        "interval_id": np.tile(np.arange(n_bins), n_rows),
        col: binned.ravel(),
    })


def exclude_outliers(s, lower_q=0.01, upper_q=0.99):
    """Set values outside the [lower_q, upper_q] quantiles to NaN."""
    lower, upper = s.quantile(lower_q), s.quantile(upper_q)
    return s.where((s >= lower) & (s <= upper))


def fit_gpboost_memorability(interval_data, outcome_col, bin_id=None):
    """LME for one bin: fixed intercept + memorability; random intercept and memorability slope per contact."""
    y = interval_data[outcome_col].values
    X = np.column_stack((np.ones(len(interval_data)), interval_data["memorability"].values))
    group_ids = pd.factorize(interval_data["subject_electrode"])[0]

    try:
        model = gpb.GPModel(group_data=group_ids,
                            group_rand_coef_data=interval_data["memorability"].values,
                            ind_effect_group_rand_coef=[1])
        model.fit(y=y, X=X)
        coef = model.get_coef(std_err=True, format_pandas=False)
        beta, se = coef[0][1], coef[1][1]   # index 1 = memorability (0 = intercept)
        z = beta / se
        return {"bin_number": bin_id, "beta": beta, "se": se, "z": z,
                "pval": norm.sf(np.abs(z)) * 2, "n": len(y), "converged": True}
    except Exception as e:
        print(f"Bin {bin_id}: model failed ({e})")
        return {"bin_number": bin_id, "beta": 0.0, "se": np.nan, "z": np.nan,
                "pval": 1.0, "n": len(y), "converged": False}


def fit_time_resolved(full_data, col, include_pre_stimulus=False):
    """Fit one model per 20-ms bin and FDR-correct the memorability p-values across bins."""
    long = bin_feature(full_data, col, include_pre_stimulus)
    rows = []
    for interval, interval_data in long.groupby("interval_id", sort=True):
        interval_data = interval_data[["memorability", "subjectID", "subject_electrode", col]].copy()
        interval_data[col] = exclude_outliers(interval_data[col])
        interval_data = interval_data.dropna()
        rows.append(fit_gpboost_memorability(interval_data, col, bin_id=interval))

    res = pd.DataFrame(rows)
    res["p_fdr"] = multipletests(res["pval"], method="fdr_bh")[1]
    t0 = -200 if include_pre_stimulus else 0
    res["start_ms"] = t0 + 20 * res["bin_number"]
    res["end_ms"] = res["start_ms"] + 20
    return res

In [ ]:
# column -> fit settings and figure settings
MEASURES = {
    "lfp":            dict(pre=True,  ylabel="LFP (µV)",                  file="fig2_erp"),
    "power_low":      dict(pre=False, ylabel="~3.5 Hz Power (dB)",        file="fig3a_slow_theta_power",
                           legend="lower right", stats_legend="lower left"),
    "power_high":     dict(pre=False, ylabel="~7 Hz Power (dB)",          file="fig3b_fast_theta_power",
                           stats_legend="upper left"),
    "phaselock_low":  dict(pre=False, ylabel="~3.5 Hz Clustering",        file="fig4a_slow_theta_clustering",
                           stats_legend="upper left"),
    "phaselock_high": dict(pre=False, ylabel="~7 Hz Clustering",          file="fig4b_fast_theta_clustering"),
    "speed_low":      dict(pre=False, ylabel="~3.5 Hz Phase Speed (Hz)",  file="fig5a_slow_theta_speed"),
    "speed_high":     dict(pre=False, ylabel="~7 Hz Phase Speed (Hz)",    file="fig5b_fast_theta_speed",
                           stats_legend="upper right"),
}

skipped = [col for col in MEASURES if col not in full_data.columns]
if skipped:
    print(f"Not in the data, skipped: {skipped}")
MEASURES = {col: spec for col, spec in MEASURES.items() if col in full_data.columns}

results = {}
for col, spec in MEASURES.items():
    print(f"Fitting {col} ...")
    results[col] = fit_time_resolved(full_data, col, include_pre_stimulus=spec["pre"])

all_results = pd.concat([res.assign(measure=col) for col, res in results.items()], ignore_index=True)
all_results.to_csv(config.RESULTS_DIR / "gpboost_results.csv", index=False)

## Significant time windows (FDR-corrected p < 0.05)

In [ ]:
def significant_windows(res, alpha=ALPHA):
    """Merge consecutive significant bins into time windows."""
    windows, current = [], None
    for _, row in res.iterrows():
        if row["p_fdr"] < alpha:
            if current is not None and row["start_ms"] == current["end_ms"]:
                current["end_ms"] = row["end_ms"]
                current["min_p_fdr"] = min(current["min_p_fdr"], row["p_fdr"])
                current["betas"].append(row["beta"])
            else:
                current = {"start_ms": row["start_ms"], "end_ms": row["end_ms"],
                           "min_p_fdr": row["p_fdr"], "betas": [row["beta"]]}
                windows.append(current)
        else:
            current = None
    for w in windows:
        w["direction"] = "positive" if np.mean(w.pop("betas")) > 0 else "negative"
    return pd.DataFrame(windows, columns=["start_ms", "end_ms", "direction", "min_p_fdr"])


windows = pd.concat([significant_windows(res).assign(measure=col) for col, res in results.items()],
                    ignore_index=True)
windows.to_csv(config.RESULTS_DIR / "significant_windows.csv", index=False)
windows

## Figures

Top panels: mean ± 95% CI for the 25% most (Q4, green) and least (Q1, blue) memorable trials, using the 25th
and 75th percentiles of memorability across all contact-trials. For the ERP, trials are baseline-corrected
(−200–0 ms), averaged per contact, and the CI is computed across contacts; for the other measures the CI is
computed across contact-trials. Phase speed is shown as means of 10-sample (~20-ms) segments.
Bottom panels: memorability coefficients (β) of the models; dark bars with a black line are significant.

In [ ]:
sns.set_theme(style="whitegrid", rc={"xtick.bottom": True, "ytick.left": True,
                                     "axes.spines.right": True, "font.size": 12})
GREEN, BLUE = "#2E8B57", "#4178C1"
BAR_WIDTH = 20  # ms

q75 = full_data["memorability"].quantile(0.75)
q25 = full_data["memorability"].quantile(0.25)
is_high = full_data["memorability"] > q75
is_low = full_data["memorability"] < q25


def mean_ci(mat):
    return mat.mean(0), 1.96 * mat.std(0, ddof=1) / np.sqrt(len(mat))


def style_axis(ax):
    ax.xaxis.set_tick_params(width=1.5, color="lightgray")
    ax.yaxis.set_tick_params(width=1.5, color="lightgray")
    ax.grid(False)


def quartile_traces(col):
    """High- and low-memorability traces (rows) for the top panel."""
    if col == "lfp":
        lfp = np.vstack(full_data["lfp"].values)[:, BASELINE_START:]
        lfp = lfp - lfp[:, :ONSET - BASELINE_START].mean(axis=1, keepdims=True)
        per_trial = full_data[["subjectID", "electrode"]].assign(lfp=list(lfp))

        def contact_means(mask):
            return np.array([np.vstack(g["lfp"].values).mean(0)
                             for _, g in per_trial[mask].groupby(["subjectID", "electrode"])])
        return contact_means(is_high), contact_means(is_low)

    high = np.vstack(full_data.loc[is_high, col].values)[:, ONSET:]
    low = np.vstack(full_data.loc[is_low, col].values)[:, ONSET:]
    if col.startswith("speed"):  # 10-sample means for display
        high = high[:, :510].reshape(len(high), 51, 10).mean(axis=2)
        low = low[:, :510].reshape(len(low), 51, 10).mean(axis=2)
    return high, low


def plot_measure(col, res, ylabel, filename, legend="upper right", stats_legend="lower right"):
    is_erp = col == "lfp"
    t0 = -200 if len(res) == 60 else 0

    high, low = quartile_traces(col)
    mean_high, ci_high = mean_ci(high)
    mean_low, ci_low = mean_ci(low)
    times = np.linspace(t0, 1000, high.shape[1])

    fig, (ax_data, ax_stat) = plt.subplots(
        2, 1, sharex=True, figsize=(6, 5) if is_erp else (6, 3),
        gridspec_kw={"height_ratios": [4, 1] if is_erp else [2, 1], "hspace": 0.0})

    # data panel
    style_axis(ax_data)
    ax_data.plot(times, mean_high, color=GREEN, label="High (Q4)")
    ax_data.fill_between(times, mean_high - ci_high, mean_high + ci_high, color=GREEN, alpha=0.35, linewidth=0)
    ax_data.plot(times, mean_low, color=BLUE, label="Low (Q1)")
    ax_data.fill_between(times, mean_low - ci_low, mean_low + ci_low, color=BLUE, alpha=0.35, linewidth=0)
    if is_erp:
        ax_data.axvline(0, color="lightgray", linestyle="--", linewidth=1.5, zorder=0)
        ax_data.axhline(0, color="lightgray", linestyle="--", linewidth=1.5, zorder=0)
    ax_data.set_ylabel(ylabel)
    ax_data.set_xlim(t0, 1000)
    ax_data.legend(frameon=False, loc=legend)

    # stats panel
    style_axis(ax_stat)
    if is_erp:
        ax_stat.axvline(0, color="lightgray", linestyle="--", linewidth=1.5, zorder=0)
    significant = (res["p_fdr"] < ALPHA).to_numpy()
    centres = np.arange(t0, 1000, 20) + BAR_WIDTH / 2
    ax_stat.bar(centres, res["beta"].astype(float), width=BAR_WIDTH, align="center", edgecolor="none",
                color=np.where(significant, "darkgray", "lightgrey"))
    for t in centres[significant]:
        ax_stat.plot([t - BAR_WIDTH / 2, t + BAR_WIDTH / 2], [0, 0], color="black", linewidth=3)
    ax_stat.set_xlabel("Time (ms)")
    ax_stat.set_ylabel("β (Memorability)" if is_erp else "β")
    ax_stat.set_xticks(np.arange(t0, 1001, 100))
    ax_stat.set_xlim(t0, 1000)
    ax_stat.legend(handles=[plt.Line2D([0], [0], color="black", lw=3, label=f"p < {ALPHA}"),
                            plt.Line2D([0], [0], color="lightgrey", lw=3, label="n.s.")],
                   loc=stats_legend, frameon=False)

    fig.align_ylabels()
    fig.tight_layout()
    for ext in ("svg", "png"):
        fig.savefig(config.FIGURES_DIR / f"{filename}.{ext}", format=ext, dpi=300)
    plt.show()


for col, spec in MEASURES.items():
    plot_measure(col, results[col], spec["ylabel"], spec["file"],
                 legend=spec.get("legend", "upper right"),
                 stats_legend=spec.get("stats_legend", "lower right" if col != "lfp" else "lower left"))